# Microsoft Agent Framework — Lab 2: Advanced Features

Building on Lab 1, this lab covers:
1. **Structured Outputs** — forcing the model to return a typed Pydantic object
2. **Multi-turn Conversations** — sessions that retain chat history
3. **Image inputs** — multi-modal messages
4. **Multi-agent coordination** — two agents collaborating in sequence
5. **MCP Tools** — the Model Context Protocol

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [1]:
from agent_framework.openai import OpenAIChatCompletionClient
from IPython.display import display, Markdown

client = OpenAIChatCompletionClient(model="gpt-4o-mini")

---
## Part 1 — Structured Outputs

Pass `output_type=MyModel` to constrain the agent's response to a Pydantic schema.  
The result is a fully typed object — no parsing needed.

In [2]:
from pydantic import BaseModel, Field
from typing import Literal


class FlightReview(BaseModel):
    airline: str = Field(description="Name of the airline")
    overall_rating: Literal["excellent", "good", "average", "poor"] = Field(
        description="Overall rating of the flight experience"
    )
    summary: str = Field(description="Brief one-sentence summary of the review")
    would_recommend: bool = Field(description="Whether the reviewer would recommend this airline")

In [3]:
review_agent = client.as_agent(
    name="review_agent",
    instructions="You analyze flight reviews and extract structured information.",
    default_options={"response_format": FlightReview},
)

review_text = """
I flew British Airways from JFK to Heathrow last month. The flight was on time,
the crew were incredibly professional and the food was surprisingly tasty.
My only complaint is that the seat was a bit cramped in economy. Overall a solid choice.
"""

result = await review_agent.run(review_text)
review: FlightReview = FlightReview.model_validate_json(result.text)
print(f"Airline:      {review.airline}")
print(f"Rating:       {review.overall_rating}")
print(f"Summary:      {review.summary}")
print(f"Recommend:    {review.would_recommend}")

Airline:      British Airways
Rating:       good
Summary:      The flight was timely with professional crew and good food, though economy seats were cramped.
Recommend:    True


---
## Part 2 — Multi-turn Conversations (Sessions)

A `session` is a conversation state container.  
Pass the same session across multiple `.run()` calls and the agent remembers context.

In [4]:
concierge = client.as_agent(
    name="travel_concierge",
    instructions=(
        "You are a friendly travel concierge. "
        "Help users plan their trips and remember their preferences."
    ),
)

session = concierge.create_session()

r1 = await concierge.run("Hi! My name is Alex and I hate long layovers.", session=session)
print("Agent:", r1)

Agent: Hi Alex! It's great to meet you. I completely understand the dislike for long layovers; they can really put a damper on travel plans. Do you have a specific destination in mind for your trip? I can help you find options with shorter layovers or direct flights if possible!


In [5]:
r2 = await concierge.run("I'm thinking of flying from New York to Tokyo. Any tips?", session=session)
print("Agent:", r2)

Agent: Absolutely, Alex! Here are some tips for your trip from New York to Tokyo:

1. **Direct Flights**: Look for direct flights to minimize your travel time. Airlines like Japan Airlines and All Nippon Airways (ANA) often operate non-stop flights from New York (JFK or Newark) to Tokyo (Narita or Haneda).

2. **Flight Timing**: Try to choose flights that leave in the evening. This way, you can sleep during the flight and arrive in Tokyo in the morning, which helps adjust to the time zone more smoothly.

3. **Airports in Tokyo**: Check if you'd prefer landing at Narita International Airport (NRT) or Haneda Airport (HND). Haneda is closer to the city center and generally more convenient for travelers.

4. **Travel Insurance**: Consider purchasing travel insurance, especially for international trips. It can help protect against unexpected events.

5. **Jet Lag**: Since there's a significant time difference, try to adjust your sleep schedule a few days before you leave by going to bed and

In [6]:
# The agent still knows who we are and what we dislike
r3 = await concierge.run("What was my main travel constraint again?", session=session)
print("Agent:", r3)

Agent: Your main travel constraint is that you hate long layovers. I'll keep that in mind when planning your trips! If you have any other preferences or constraints, feel free to share!


---
## Part 3 — Image Inputs (Multi-modal)

Microsoft Agent Framework supports multi-modal messages.  
Pass image content alongside text to vision-capable models.

In [7]:
from io import BytesIO
import base64
import requests

image_url = "https://edwarddonner.com/wp-content/uploads/2024/10/from-software-engineer-to-AI-DS.jpeg"
image_bytes = requests.get(image_url).content
image_b64 = base64.b64encode(image_bytes).decode()

print(f"Image loaded: {len(image_bytes):,} bytes")

Image loaded: 846,600 bytes


In [8]:
# The Agent Framework accepts base64-encoded images inline
vision_agent = client.as_agent(
    name="vision_agent",
    instructions="You are an expert at describing and analysing images.",
)

# Build a message list with both text and image content
from agent_framework import Content
import base64

multimodal_input = [
    Content.from_text(text="Describe this image in detail."),
    Content.from_data(data=base64.b64decode(image_b64), media_type="image/jpeg"),
]

result = await vision_agent.run(multimodal_input)
display(Markdown(str(result)))

The image depicts a vibrant, stylized room that suggests a blend of technology and creativity. 

1. **Room Layout**: The interior features a light blue wall with a large open door on the right. The door leads to a burst of colorful energy, depicted as a radiant explosion of stripes in various colors, including reds, yellows, greens, and blues. This colorful backdrop contrasts sharply with the room's muted palette.

2. **Furniture**: In the foreground, there is a wooden desk with an ergonomic chair positioned in front of it. On the desk, a computer monitor displays lines of code in white text on a black background, suggesting programming work is ongoing. Next to the monitor is a lamp providing focused light.

3. **Technology and Decor**: A small, friendly-looking robot sits on the desk, emphasizing the tech theme. There are also two speakers beside the computer. The floor features colorful streaks leading towards the door, further enhancing the dynamic feel of the space.

4. **Windows and Lighting**: The room has a window on the left side, through which natural light enters, casting shadows on the wall. The clock on the wall indicates it’s a little past 10 o'clock, adding a subtle touch to the atmosphere of the space.

5. **Other Elements**: There is a small box on the floor and a cabinet with drawers beside the desk, contributing to the functional and organized aesthetic of the workspace.

Overall, the image evokes a sense of innovation and excitement, merging a relatable work environment with a portal to something extraordinary, suggested by the explosion of colors emanating from the doorway.

---
## Part 4 — Multi-Agent Coordination

Here we manually orchestrate two agents: a **Researcher** and an **Evaluator**.  
The researcher gathers information, then the evaluator critiques and approves.

This is the simplest multi-agent pattern: sequential hand-off in Python code.  
In Lab 3 and 4 we'll use Workflows for more structured orchestration.

In [9]:
import sqlite3, os

DB_PATH = "tickets.db"

def get_city_price(city_name: str) -> float | None:
    """Get the round-trip ticket price to a city."""
    if not os.path.exists(DB_PATH):
        return None
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

In [10]:
researcher = client.as_agent(
    name="researcher",
    instructions=(
        "You are a travel researcher. Look up flight prices and suggest the best value destinations. "
        "Always use your tools to look up real prices from the database."
    ),
    tools=get_city_price,
)

evaluator = client.as_agent(
    name="evaluator",
    instructions=(
        "You are a senior travel editor. Review the researcher's suggestions critically. "
        "If the suggestions are good, reply with 'APPROVED: ' followed by your comments. "
        "Otherwise provide constructive feedback."
    ),
)

In [11]:
task = "Find the top 3 cheapest European destinations from our database and explain why they're good value."

print("=== Researcher ===")
research_result = await researcher.run(task)
print(research_result)

print("\n=== Evaluator ===")
eval_input = f"Here is the researcher's suggestion:\n\n{research_result}\n\nPlease review it."
eval_result = await evaluator.run(eval_input)
print(eval_result)

=== Researcher ===
It seems that there was an issue retrieving the flight price data for the destinations I queried. However, I can provide some insights based on commonly known trends and factors that make certain European destinations good value.

### 1. **Budapest, Hungary**
- **Why it's good value:** Budapest is known for its thermal baths, stunning architecture, and vibrant nightlife. The cost of living is relatively low compared to other European capitals, making dining out, public transportation, and accommodation very affordable. Many cultural attractions, like the Buda Castle and the Parliament Building, have free or low-cost entrance fees.

### 2. **Prague, Czech Republic**
- **Why it's good value:** This charming city offers a mix of history and culture with its beautiful Old Town, the iconic Charles Bridge, and the Prague Castle. The prices for food and drinks are typically cheaper than in Western Europe, and the public transport system is efficient and budget-friendly. Man

In [12]:
# Keep iterating until the evaluator approves
max_turns = 5
research_session = researcher.create_session()
eval_session = evaluator.create_session()

current_research = await researcher.run(task, session=research_session)
print(f"Turn 1 — Researcher:\n{current_research}\n")

for turn in range(2, max_turns + 1):
    eval_input = f"Research output:\n{current_research}\n\nReview this. Reply APPROVED if it's good."
    feedback = await evaluator.run(eval_input, session=eval_session)
    print(f"Turn {turn} — Evaluator:\n{feedback}\n")

    if "APPROVED" in str(feedback).upper():
        print("✓ Evaluator approved the research!")
        break

    current_research = await researcher.run(
        f"The evaluator said: {feedback}. Please revise your suggestions.",
        session=research_session,
    )
    print(f"Turn {turn} — Researcher (revised):\n{current_research}\n")

Turn 1 — Researcher:
It seems that I couldn't retrieve the flight prices for the suggested European destinations at this moment. However, I can still suggest three popular European destinations that are generally regarded as affordable, along with reasons why they offer good value:

1. **Budapest, Hungary**:
   - **Affordability**: Budapest is known for its low cost of living compared to other European capitals, making it budget-friendly for travelers.
   - **Rich Culture**: The city boasts stunning architecture, a vibrant arts scene, and historic sites like the Buda Castle and the Parliament building.
   - **Thermal Baths**: Budapest is famous for its thermal baths, providing a unique and relaxing experience.
   - **Delicious Cuisine**: Enjoy local dishes like goulash and chimney cakes at affordable prices.

2. **Prague, Czech Republic**:
   - **Cost-Effective**: Accommodations, food, and attractions are often less expensive than in Western European cities.
   - **Beautiful Architectu

### Agent as Tool

The manual hand-off above (call Researcher → pass text to Evaluator) is straightforward but brittle: the orchestrating code knows too much about each agent's role.

A cleaner approach: **convert an agent into a tool** with `.as_tool()`. The outer agent then decides *when* to call the inner agent, passing a natural-language task string — just like any other function tool.


In [13]:
# Convert the researcher into a tool for the evaluator to call directly
researcher_tool = researcher.as_tool(
    name="consult_researcher",
    description=(
        "Ask the travel researcher to look up destination prices and provide recommendations. "
        "Pass the full task as a natural-language string."
    ),
    arg_name="task",
)

# Evaluator-as-orchestrator: it decides when to ask the researcher
orchestrator = client.as_agent(
    name="orchestrator",
    instructions=(
        "You are a senior travel editor. Your job:\n"
        "1. Delegate the research to consult_researcher.\n"
        "2. Review what comes back critically.\n"
        "3. If it's good, output 'APPROVED: ...' with your comments.\n"
        "4. If it needs improvement, call the researcher again with specific feedback."
    ),
    tools=[researcher_tool],
)

result = await orchestrator.run(
    "Find the top 3 cheapest European destinations and explain their value."
)
print(result)


APPROVED: The research provides a comprehensive overview of the top three cheapest European destinations for 2024, highlighting Budapest, Prague, and Lisbon. Each destination is well-supported with details on accommodation, meal costs, and attractions. 

- **Budapest** offers great value with its range of affordable accommodation and inexpensive meals, along with many free or low-cost attractions. The inclusion of transportation costs is also a strong point.
  
- **Prague** highlights its historical allure with budget-friendly options for both lodging and meals, making it accessible for travelers on a budget.
  
- **Lisbon** is positioned as slightly higher on flight prices, but the overall affordability in accommodations and meals adds to its value proposition. It's good to see the emphasis on attractions that are either free or low-cost.

Overall, the analysis effectively captures the essence of each destination's appeal in terms of budget travel. 


---
## Part 5 — MCP Tools

The Microsoft Agent Framework natively supports the **Model Context Protocol**.  
MCP tools are server processes that expose tools to agents.

> **Windows users:** MCP stdio servers require WSL. See `setup/SETUP-WSL.md`.

In [16]:
from agent_framework import MCPStdioTool

fetch_tool = MCPStdioTool(
    name="fetch",
    command="uvx",
    args=["mcp-server-fetch"],
    request_timeout=30,
)

async with fetch_tool:
    web_agent = client.as_agent(
        name="web_agent",
        instructions="You are a web research assistant. Use your tools to fetch and summarise web pages.",
        tools=fetch_tool,
    )

    result = await web_agent.run(
        "Fetch https://edwarddonner.com and summarise what you find. Reply in Markdown."
    )

display(Markdown(str(result)))


# Summary of Edward Donner's Website

![Edward Donner](https://i0.wp.com/edwarddonner.com/wp-content/uploads/2023/12/cropped-edworkprofile2.png?fit=1128%2C1128&ssl=1)

Welcome to the personal site of **Edward Donner**, a software developer and AI enthusiast. Here are the key points from his website:

- **Interests**: Ed enjoys writing code, experimenting with large language models (LLMs), amateur music production, and keeping up with tech news on [Hacker News](https://news.ycombinator.com/).

- **Professional Background**:
  - Co-founder and CTO of [Nebula.io](https://nebula.io/?utm_source=ed&utm_medium=referral), a company focused on utilizing AI to positively impact personal development.
  - Founder and former CEO of **untapt**, an AI startup that was acquired in 2021.

- **Teaching**: 
  - Ed has developed several popular courses on Udemy, focusing on LLMs, which have attracted over **500,000 students** from **194 countries**. The full curriculum can be explored [here](https://edwarddonner.com/curriculum/).
  
- **Engagement**: He expresses a strong passion for discussing LLMs and is excited to connect with visitors, especially those coming from his courses.

This site serves as a hub for Ed's professional pursuits and educational content in the tech field.

---
### Summary

| Feature | API |
|---|---|
| Structured outputs | `client.as_agent(..., output_type=MyModel)` |
| Multi-turn session | `session = agent.create_session()` then `agent.run(..., session=session)` |
| Multi-modal input | `[TextContent(...), ImageContent(...)]` |
| Multi-agent hand-off | plain Python: call agents sequentially |
| Agent as Tool | `agent.as_tool(name=..., arg_name=...)` |
| MCP tools | `get_mcp_tools(McpServerParams(...))` |

In **Lab 3** we'll use the Workflow engine for structured multi-step pipelines.